[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione/blob/main/02_primo_scout_automatico.ipynb)

# Il primo scout automatico

Nel primo notebook abbiamo guardato i dati. Ora costruiamo il primo modello predittivo: una formula semplice che stima il valore di mercato a partire da una sola informazione, l'`overall`.

> **Missione** — Costruire una prima regressione lineare: una retta che usa l'overall per prevedere il valore di mercato.


## Riapriamo il file

Nel primo notebook abbiamo *guardato* i dati: ora vogliamo costruire qualcosa che li *usi*. Il direttore sportivo vuole che il nostro primo modello impari sui dati che già conosciamo bene, così potremo concentrarci sull'algoritmo senza distrarci con sorprese sul dataset. Il blocco che segue è identico a quello del primo episodio: serve solo a riportare in memoria il file e i suoi giocatori.


In [ ]:
# Librerie necessarie per scaricare ed estrarre il dataset
import urllib.request, zipfile
from pathlib import Path

# Librerie per l'analisi dei dati e la visualizzazione
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Link al dataset e percorsi locali
DATA_URL = (
    "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/"
    "fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
)
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

# Scarichiamo il dataset se non è già presente
if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

# Estraiamo il dataset se non è già stato fatto
if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

# Cerchiamo il file CSV dei giocatori dentro la cartella estratta
csv_files = (
    list(DATA_DIR.rglob("players_22.csv"))
    or list(DATA_DIR.rglob("*players*.csv"))
)

# Carichiamo il CSV in un DataFrame pandas: ogni riga = un giocatore
raw_data = pd.read_csv(csv_files[0], low_memory=False)
print(f"Caricato: {raw_data.shape[0]} giocatori, {raw_data.shape[1]} colonne")
raw_data.head()


Stessa pulizia del primo notebook: teniamo solo le colonne utili, eliminiamo le righe senza valore di mercato e tagliamo via l'1% di giocatori più costosi. È una scelta consapevole: stiamo studiando il mercato dei calciatori *tipici*, non quello dei fuoriclasse.


In [ ]:
# Definiamo un sottoinsieme di colonne che ci interessano
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

# Prendiamo solo le colonne che ci interessano e copiamo il dataset
# per lavorarci sopra senza modificare l'originale
dataset = raw_data[wanted_columns].copy()

# Manteniamo solo i giocatori con valore noto e positivo
dataset = dataset.dropna(subset=["value_eur", "age", "overall", "potential"])
dataset = dataset[dataset["value_eur"] > 0]

# Riduciamo l'effetto dei super-giocatori (l'1% più costoso)
dataset = dataset[dataset["value_eur"] <= dataset["value_eur"].quantile(0.99)]

# Per leggibilità usiamo il valore in milioni di euro
dataset["value_mln_eur"] = dataset["value_eur"] / 1_000_000
if "wage_eur" in dataset.columns:
    dataset["wage_k_eur"] = dataset["wage_eur"] / 1_000

print("Forma del dataset pulito:", dataset.shape)
dataset.head(10)


> **Regressione** — Un problema di regressione consiste nel prevedere un numero. Qui il numero e' il valore di mercato del calciatore. Il modello riceve in ingresso una o piu' caratteristiche e restituisce una previsione numerica.


## Prepariamo input e target

## Prepariamo input e target

Da scout esperti sappiamo già che l'overall era la variabile più correlata col valore di mercato. Costruiamo allora il modello più semplice possibile: una formula che, dato un solo indizio (l'overall), restituisce una stima del valore in milioni di euro. Ma prima di addestrarla dobbiamo separare con chiarezza due ruoli: le colonne usate come **indizi** (gli *input*) e la colonna che vogliamo **prevedere** (il *target*).

Questa distinzione sembra banale, ma è la stessa ogni volta che si fa machine learning: si dice all'algoritmo *cosa guardare* e *cosa indovinare*.


In [ ]:
# Importiamo gli strumenti del machine learning che useremo:
# - LinearRegression: il modello a retta
# - le tre metriche per misurare quanto sbaglia
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)

# X = matrice degli input. Per ora una sola colonna: l'overall.
# Le doppie parentesi quadre servono perché sklearn vuole una matrice
# (anche se ha una sola colonna), non una serie monodimensionale.
X = dataset[["overall"]]

# y = il target da prevedere, il valore di mercato in milioni
y = dataset["value_mln_eur"]

X.head(), y.head()


> **Input e target** — Nel linguaggio del machine learning, gli input sono le informazioni che forniamo al modello. Il target e' cio' che vogliamo prevedere. In questo caso l'input e' overall, il target e' value_mln_eur.


### La forma del modello

Prima di lasciare lavorare l'algoritmo, chiediamoci che cosa stiamo davvero cercando. Quando l'input è una sola variabile $x$, la regressione lineare ipotizza che il legame col target abbia la forma più semplice immaginabile: una **retta**, di equazione

$\displaystyle \hat{y} \;=\; w \cdot x + b$

dove:

- $\hat{y}$ (si legge "y-cappello") è la **previsione** del modello.
- $w$ è la **pendenza**: di quanto cresce la previsione se l'input aumenta di $1$.
- $b$ è l'**intercetta**: il valore di $\hat{y}$ quando $x = 0$ (un'estrapolazione astratta, ma serve a fissare la retta).

Per un giocatore reale con valore $y_i$, l'errore commesso dal modello è la differenza tra ciò che osserviamo e ciò che il modello prevede, il **residuo**:

$\displaystyle r_i \;=\; y_i - \hat{y}_i$

Lo scopo dell'algoritmo è scegliere $w$ e $b$ in modo che, *complessivamente*, i residui siano piccoli. Ma cosa significa "complessivamente piccoli"? È la prossima domanda.


## Alleniamo il modello

Adesso lasciamo che l'algoritmo trovi da solo la retta migliore. Quando avrà finito leggeremo i due numeri che ha imparato — pendenza e intercetta — e li scriveremo in chiaro: questa è letteralmente la *ricetta* del nostro primo scout automatico.


In [ ]:
# Creiamo il modello (per ora vuoto: non sa ancora nulla)
model = LinearRegression()

# .fit() è il cuore dell'addestramento: l'algoritmo cerca i valori
# di w e b che minimizzano l'errore medio sui dati che gli passiamo.
model.fit(X, y)

# Estraiamo i parametri imparati: il coefficiente (la pendenza w)
# e l'intercetta b.
w = model.coef_[0]
b = model.intercept_
print(f"Formula imparata: valore ≈ {w:.2f} * overall + ({b:.2f})")


> **Parametri imparati** — La retta ha due parametri: pendenza w e intercetta b. All'inizio non sappiamo quali valori usare. L'algoritmo li sceglie cercando la retta che riduce l'errore medio sui dati disponibili.


### La funzione di costo

L'algoritmo "trova la retta migliore", ma per farlo ha bisogno di un criterio numerico: "errore complessivamente piccolo" deve diventare un *numero da minimizzare*. La regressione lineare usa l'**errore quadratico medio** (MSE):

$\displaystyle \mathrm{MSE}(w, b) \;=\; \frac{1}{n} \sum_{i=1}^{n} \bigl(y_i - (w \cdot x_i + b)\bigr)^2$

Perché proprio il **quadrato** dell'errore?

- Così gli errori positivi e negativi non si cancellano fra loro: tutti contribuiscono in positivo.
- Gli errori grandi pesano molto di più degli errori piccoli, quindi il modello viene "punito" severamente quando sbaglia tanto su un giocatore.

L'algoritmo cerca la coppia $(w, b)$ che rende l'$\mathrm{MSE}$ il più piccolo possibile. Per la regressione lineare esiste una **formula chiusa** (il *metodo dei minimi quadrati*) che la calcola in un colpo solo: niente tentativi, niente iterazioni — è pura algebra lineare. Dietro al `.fit()` di sklearn, è esattamente questa la matematica che gira.


## Visualizziamo la retta dello scout automatico

Per capire cosa ha imparato il modello, disegniamolo *in azione*: i puntini sono i giocatori reali, la retta è la previsione del nostro scout automatico al variare dell'overall. Se il modello ha imparato bene, la retta dovrebbe attraversare la nuvola dei punti seguendone la tendenza generale.


In [ ]:
# Generiamo 100 punti equispaziati lungo l'asse degli overall:
# ci servono per disegnare la retta del modello in modo continuo.
x_line = np.linspace(
    dataset["overall"].min(), dataset["overall"].max(), 100
).reshape(-1, 1)

# Per ognuno di quei punti chiediamo al modello la sua previsione
y_line = model.predict(x_line)

# Disegniamo la nuvola dei giocatori reali e sopra la retta del modello
plt.figure(figsize=(8,5))
plt.scatter(
    dataset["overall"], dataset["value_mln_eur"],
    alpha=0.22, label="Giocatori"
)
plt.plot(x_line, y_line, linewidth=3, label="Modello lineare")
plt.xlabel("Overall rating")
plt.ylabel("Valore [milioni di euro]")
plt.title("Prima regressione: valore previsto da overall")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Dove sbaglia il modello? I residui


Una retta che passa nel mezzo della nuvola sembra ragionevole, ma per giudicarla davvero non basta uno sguardo d'insieme: dobbiamo guardare gli **errori giocatore per giocatore**. Per ogni calciatore reale calcoliamo la differenza tra il valore vero e la previsione del modello, e disegniamo questi residui in funzione dell'overall. La forma che assumeranno ci dirà *dove* il modello sbaglia in modo sistematico.


In [ ]:
# Per ogni giocatore calcoliamo il residuo:
# residuo = valore reale - valore previsto
# Un residuo positivo significa che il modello ha sottostimato il giocatore;
# negativo, che lo ha sovrastimato.
residui = y - model.predict(X)

# Disegniamo i residui in funzione dell'overall
plt.figure(figsize=(8,4))
plt.scatter(dataset["overall"], residui, alpha=0.25)

# La linea rossa segna l'errore zero: la previsione perfetta.
# Punti sopra = sottostime, punti sotto = sovrastime.
plt.axhline(0, color="red", linestyle="--", linewidth=2)
plt.xlabel("Overall rating")
plt.ylabel("Residuo [milioni di euro]")
plt.title("Residui del modello: dove e di quanto sbaglia")
plt.grid(True, alpha=0.3)
plt.show()


### Teoria — Come si leggono i residui

Il grafico dei residui mostra l'errore $r_i = y_i - \hat{y}_i$ in funzione dell'input.

- Se i residui sono **sparsi attorno a zero** senza struttura, il modello sta lavorando bene.
- Se mostrano un **andamento** (curva, ventaglio, gruppi), allora c'è informazione che la retta non riesce a catturare: forse serve un modello più ricco.
- Residui **molto grandi** isolati segnalano giocatori "particolari" (outlier) che meriterebbero un'analisi a parte.


---

> **Domanda** — Secondo voi la retta sottostima o sovrastima i campioni? E cosa succede ai giocatori fortissimi, con overall molto alto?


## Misuriamo l'errore

Guardando i residui abbiamo un'idea qualitativa di dove il modello cade. Ma *in generale*, quanto sbaglia? Per confrontare modelli diversi (lo faremo nei prossimi episodi) serve riassumere le sue prestazioni in pochi numeri sintetici. Calcoliamo le tre misure standard della regressione: ognuna racconta una sfumatura diversa dell'errore.


In [ ]:
# Calcoliamo la previsione del modello per ogni giocatore del dataset
y_pred = model.predict(X)

# MAE: errore medio assoluto, in milioni di euro.
# Intuitivo: "in media il modello sbaglia di X milioni".
mae = mean_absolute_error(y, y_pred)

# RMSE: radice dell'errore quadratico medio.
# Penalizza di più gli errori grandi rispetto al MAE.
rmse = mean_squared_error(y, y_pred) ** 0.5

# R^2: frazione di variabilità del target spiegata dal modello.
# Vale 1 se le previsioni sono perfette, 0 se non fa meglio della media.
r2 = r2_score(y, y_pred)

print(f"Errore medio assoluto: {mae:.2f} milioni di euro")
print(f"RMSE: {rmse:.2f} milioni di euro")
print(f"R^2: {r2:.3f}")


> **Errore medio assoluto** — Se l'errore medio assoluto vale, per esempio, 3 milioni, significa che in media le previsioni sbagliano di circa 3 milioni di euro. Questa misura e' intuitiva perche' ha la stessa unita' di misura del target.


### Teoria — Tre metriche per giudicare un modello

**MAE** (errore medio assoluto): media dei moduli dei residui. Stessa unità di misura del target.

$\displaystyle \mathrm{MAE} \;=\; \frac{1}{n} \sum_{i=1}^{n} \bigl|\, y_i - \hat{y}_i \,\bigr|$

**RMSE** (radice dell'errore quadratico medio): più severo del MAE perché penalizza di più gli errori grandi.

$\displaystyle \mathrm{RMSE} \;=\; \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$

**$R^2$** (coefficiente di determinazione): la frazione della variabilità del target spiegata dal modello.

$\displaystyle R^2 \;=\; 1 \;-\; \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$

- $R^2 = 1$: previsioni perfette.
- $R^2 = 0$: il modello non fa meglio che indovinare sempre la media.
- $R^2 < 0$: il modello fa *peggio* della media (succede davvero, sui dati nuovi).


## Facciamo scouting su alcuni giocatori

Abbiamo numeri aggregati, ma per *sentire* davvero quanto è bravo lo scout serve guardarlo all'opera caso per caso. Estraiamo dieci giocatori a caso e per ognuno mettiamo a confronto il valore reale, la previsione del modello e l'errore. Quanto si avvicina, giocatore per giocatore?


In [ ]:
# Estraiamo 10 giocatori a caso (random_state fissa il seed:
# tutti gli studenti vedranno gli stessi 10 nomi)
sample = dataset.sample(10, random_state=7).copy()

# Chiediamo al modello la sua previsione per ognuno
sample["predicted_value_mln_eur"] = model.predict(sample[["overall"]])

# Calcoliamo l'errore: previsione - valore reale
# (segno positivo = il modello ha sovrastimato)
sample["error_mln_eur"] = (
    sample["predicted_value_mln_eur"] - sample["value_mln_eur"]
)

# Mostriamo la tabella riassuntiva, arrotondata a 2 decimali
sample[[
    "short_name", "age", "overall", "potential",
    "value_mln_eur", "predicted_value_mln_eur", "error_mln_eur"
]].round(2)


---

> **Cosa dovremmo aver capito** — Abbiamo costruito un primo modello molto semplice. Funziona meglio del tirare a caso, ma e' limitato: uno scout vero non guarda solo l'overall. Nel prossimo notebook aggiungeremo altri indizi.
